# AdLens: data exploration

Run from the project root with `ADLENS_DB_PASSWORD` set (or a `.env`). All numbers come from MySQL via the same tools the agents use.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from tools import database_tools as T, metric_tools as M, simulation_tools as SIM

## 1. What data exists (see `results/data_profile.md` from `data/profile_data.py`)

In [ ]:
prof = json.load(open('../results/data_profile.json'))
pd.DataFrame([dict(table=p['name'], rows=p['rows'], provenance=p['provenance_guess'], ad_data=p['has_impressions_and_clicks']) for p in prof])

## 2. The real orders: daily delivery time alternates between ~23 and ~30 minutes

In [ ]:
ts = pd.DataFrame(T.get_time_series('orders', 'avg_delivery_time', '2022-03-01', '2022-04-06')['points'])
ts.plot(x='period', y='value', marker='o', figsize=(10, 3), title='avg delivery time per day (REAL)')

## 3. Why? Shift-share decomposition of a slow day vs the previous fast day

In [ ]:
scan = T.scan_dimensions('orders', 'avg_delivery_time', '2022-03-13', '2022-03-13', '2022-03-14', '2022-03-14')
pd.DataFrame(scan['dimensions'])

In [ ]:
pd.DataFrame(T.compare_periods('orders', 'avg_delivery_time', 'distance_band', '2022-03-13', '2022-03-13', '2022-03-14', '2022-03-14')['segments'])

Distance bands do not overlap at all between the two days: every order on the fast day is under 10 km, every order on the slow day is 10 km or more. The pattern repeats across the whole dataset, which points to how the Kaggle dataset was constructed rather than to delivery operations.

In [ ]:
SIM.simulate_mix('orders', 'avg_delivery_time', 'distance_band', '2022-03-13', '2022-03-13', '2022-03-14', '2022-03-14')

## 4. The synthetic ad layer (restaurant_ads): every CTR/CVR/CPC here is SYNTHETIC

In [ ]:
pd.DataFrame(T.get_segment_metrics('restaurant_ads', 'ctr', 'placement', '2024-01-01', '2024-01-07')['segments'])